# 02 · Experiments

3 representations × 3 architectures, each evaluated with k-fold cross-validation.
Every run saves its fold scores to `results/<architecture>__<representation>.json`.

In [ ]:
# Setup: make the course code (part1) and our code (src) importable.
# Run notebooks from this folder (JupyterLab / VS Code do this by default).
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

%load_ext autoreload
%autoreload 2

In [ ]:
from part1 import experiment
from src import config, data, encoders, results_io

## Data and folds

In [ ]:
pairs, outcome_space = data.load_pairs("train")
folds = experiment.cross_validation_split(pairs, k=config.K_FOLDS, seed=config.SEED)

## Tokeniser

Reuse the tutorial's or train our own? On what corpus, which vocab size?
Note the choice + reason in the decisions log.

In [ ]:
corpus = data.tokenizer_corpus()
tokenizer = experiment.train_tokenizer(corpus, vocab_size=10_000, min_frequency=2)  # TODO: tune
tokenizer.get_vocab_size()

## Representations

In [ ]:
representations = {
    "indicator": encoders.TokenIndicator(tokenizer),
    "bow": encoders.BagOfWords(tokenizer),
    # "ours": encoders.ThirdRepresentation(tokenizer),
}

## Log-linear

In [ ]:
for rep_name, enc in representations.items():
    name = f"loglin__{rep_name}"
    out = experiment.cross_validation_experiment(
        folds, outcome_space, enc, "logistic", save_to=str(config.MODELS_DIR / name)
    )
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size())
    print(name, out["accuracies"])

## MLP (one hidden layer)

In [ ]:
smlp_args = dict(hidden_dim=128, dropout=0.1, num_epochs=10)  # TODO: tune

for rep_name, enc in representations.items():
    name = f"mlp__{rep_name}"
    out = experiment.cross_validation_experiment(
        folds, outcome_space, enc, "mlp", save_to=str(config.MODELS_DIR / name), smlp_args=smlp_args
    )
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size(), **smlp_args)
    print(name, out["accuracies"])

## Our third architecture

See `src/models.py`.

In [ ]:
# TODO